[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week08.ipynb)

# Week 8: Convolutional Networks I
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Convolution, pooling, and feature maps; building a CNN image classifier.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Build and train a CNN image classifier.
- Understand convolution, pooling, and feature maps.
- Compute how shapes and parameters change layer by layer.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Build a CNN and print the layer-by-layer output shapes.

In [ ]:
# A small CNN; print the shape after each layer
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28 -> 14
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14 -> 7
    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))
x = torch.randn(4, 1, 28, 28)
for layer in cnn:
    x = layer(x); print(f"{layer.__class__.__name__:9s}", tuple(x.shape))

## Demonstration 2
Compute output sizes and parameter counts by hand and verify against a summary.

In [ ]:
# Output size and parameter count, by formula and by code
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
print("output:", tuple(conv(torch.randn(1, 3, 32, 32)).shape), "  formula (32+2-3)/1+1 = 32")
params = sum(p.numel() for p in conv.parameters())
print("params:", params, "= (3*3*3 + 1) * 16 =", (3 * 3 * 3 + 1) * 16)

## Demonstration 3
Train on FashionMNIST and visualize a few feature maps.

In [ ]:
# Train one batch on FashionMNIST and visualize first-conv feature maps
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
dl = DataLoader(datasets.FashionMNIST("./data", train=True, download=True, transform=transforms.ToTensor()),
                batch_size=128, shuffle=True)
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(8 * 14 * 14, 10)).to(device)
opt = torch.optim.Adam(model.parameters(), 1e-3); f = nn.CrossEntropyLoss()
xb, yb = next(iter(dl)); xb, yb = xb.to(device), yb.to(device)
opt.zero_grad(); f(model(xb), yb).backward(); opt.step()
fmap = model[0](xb[:1]).detach().cpu()
fig, ax = plt.subplots(1, 4, figsize=(8, 2))
for i in range(4):
    ax[i].imshow(fmap[0, i]); ax[i].axis("off")
plt.show()

---
Student materials for this week: the lab handout (`labs/week08.html`) and the curated references (`references/week08.html`) in the course site.